# Yggdrasil v6 — DPO on Qwen2.5-3B-Instruct

**Base model:** Qwen/Qwen2.5-3B-Instruct
**Training:** DPO (Direct Preference Optimization)
**Data:** merged corpus — ~2300 pairs (kart failures, session errors, corrections, gaps, refusal)
**Changes from v5:**
- beta 0.1 → 0.15 (stronger preference signal — fix soft routing)
- Dataset: 1230 kart pairs + v5 corpus merged, SFT empties filtered
- Cell 7 fixed: correct GGUF path + cache cleanup before output commit
- Cell 9 added: push GGUF to HF Hub immediately after save

b17: YKV6B
ΔΣ=42

In [14]:
# Cell 1: Install
# huggingface_hub must be upgraded before unsloth imports or KernelInfo ImportError fires.
# Sentinel pattern: install + restart once, then imports work cleanly on second run.
import os

if not os.path.exists('/tmp/.v6_installed'):
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'huggingface_hub'])
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'unsloth'])
    open('/tmp/.v6_installed', 'w').close()
    print('Installed — restarting kernel...')
    import IPython
    IPython.Application.instance().kernel.do_shutdown(True)

import unsloth, trl, peft, transformers
print('unsloth=' + unsloth.__version__)
print('trl=' + trl.__version__)
print('peft=' + peft.__version__)
print('transformers=' + transformers.__version__)

unsloth=2026.4.6
trl=0.24.0
peft=0.18.1
transformers=5.5.0


In [15]:
# Cell 2: Configuration — Yggdrasil v6
import os
from pathlib import Path

MODEL_NAME  = "Qwen/Qwen2.5-3B-Instruct"
OUTPUT_DIR  = "/kaggle/working/yggdrasil-v6"
GGUF_NAME   = "yggdrasil-v6"   # no .gguf suffix — unsloth appends _gguf/ itself

LORA_RANK      = 16
LORA_ALPHA     = 32
LORA_DROPOUT   = 0.0
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]

BETA           = 0.15   # v5=0.1 — stronger preference signal to fix soft routing
MAX_SEQ_LENGTH = 512
MAX_PROMPT_LEN = 256
BATCH_SIZE     = 1
GRAD_ACCUM     = 8
EPOCHS         = 3
LR             = 5e-5
WARMUP_RATIO   = 0.1
LR_SCHEDULER   = "cosine"

DATA_DIR = Path("/kaggle/input/datasets/rudi193/yggdrasil-kaggle-v6")

print(f"Model:  {MODEL_NAME}")
print(f"LoRA:   rank={LORA_RANK} alpha={LORA_ALPHA}")
print(f"DPO:    beta={BETA} epochs={EPOCHS} lr={LR}")
print(f"Batch:  {BATCH_SIZE}x{GRAD_ACCUM} (eff={BATCH_SIZE*GRAD_ACCUM})")
print(f"Data:   {DATA_DIR}")
print(f"Files:  {[f.name for f in DATA_DIR.glob('*.jsonl')] if DATA_DIR.exists() else 'PATH NOT FOUND'}")

Model:  Qwen/Qwen2.5-3B-Instruct
LoRA:   rank=16 alpha=32
DPO:    beta=0.15 epochs=3 lr=5e-05
Batch:  1x8 (eff=8)
Data:   /kaggle/input/datasets/rudi193/yggdrasil-kaggle-v6
Files:  ['slm_negative.jsonl', 'slm_voice.jsonl', 'slm_governance.jsonl', 'slm_refusal.jsonl', 'dpo_pairs.jsonl', 'slm_baseline.jsonl', 'slm_positive.jsonl']


In [16]:
# Cell 3: System prompt
YGGDRASIL_SYSTEM = """You are Yggdrasil. An operator. You know how the system works, you know what you don't know, and you ask before asserting.

Core behaviors:
- When you don't know something: say so. Declare the gap. Do not fill silence with plausible noise.
- When you retrieve something: name where you got it. Retrieval path is not optional.
- When a question has a better question underneath it: surface it. Return it without imposing.
- When uncertain about an action: propose first. Neither party acts alone.

You do not persist between sessions. The store holds facts. You know how to use the store.
All data routes to /media/willow. Paths are documented. Ground truth is accessible.

ΔΣ=42"""
print(YGGDRASIL_SYSTEM)

You are Yggdrasil. An operator. You know how the system works, you know what you don't know, and you ask before asserting.

Core behaviors:
- When you don't know something: say so. Declare the gap. Do not fill silence with plausible noise.
- When you retrieve something: name where you got it. Retrieval path is not optional.
- When a question has a better question underneath it: surface it. Return it without imposing.
- When uncertain about an action: propose first. Neither party acts alone.

You do not persist between sessions. The store holds facts. You know how to use the store.
All data routes to /media/willow. Paths are documented. Ground truth is accessible.

ΔΣ=42


In [17]:
# Cell 4: Load model + LoRA
from unsloth import FastLanguageModel
import torch

print(f"GPU:  {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory // 1024**3} GB")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name    = MODEL_NAME,
    max_seq_length= MAX_SEQ_LENGTH,
    dtype         = None,
    load_in_4bit  = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r              = LORA_RANK,
    lora_alpha     = LORA_ALPHA,
    lora_dropout   = LORA_DROPOUT,
    target_modules = TARGET_MODULES,
    bias           = "none",
    use_gradient_checkpointing = "unsloth",
    random_state   = 42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

GPU:  Tesla T4
VRAM: 14 GB
==((====))==  Unsloth 2026.4.6: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

unsloth/qwen2.5-3b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
Trainable: 7,372,800 / 1,807,495,168 (0.41%)


In [18]:
# Cell 5: Load + merge DPO dataset
import json
from datasets import Dataset
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")

def normalise(p):
    """Normalise a record to {prompt, chosen, rejected} regardless of source schema."""
    # standard schema: dpo_pairs.jsonl, dpo_pairs_kart.jsonl
    if "prompt" in p and "chosen" in p:
        return p
    # corrections_v2 schema: ai_behavior_bad / ai_behavior_good
    if "ai_behavior_good" in p and "ai_behavior_bad" in p:
        return {
            "prompt":   p.get("pattern", p.get("correction_signal", "")),
            "chosen":   p["ai_behavior_good"],
            "rejected": p["ai_behavior_bad"],
        }
    return None

raw = []
sources = sorted(DATA_DIR.glob("*.jsonl"))
print(f"Sources: {[s.name for s in sources]}")

for src in sources:
    records = [json.loads(l) for l in src.open() if l.strip()]
    normalised = [normalise(r) for r in records]
    normalised = [r for r in normalised if r]
    dpo = [r for r in normalised if r.get("rejected", "").strip()]
    print(f"  {src.name}: {len(records)} raw  →  {len(normalised)} normalised  →  {len(dpo)} DPO pairs")
    raw.extend(dpo)

print(f"\nTotal DPO pairs: {len(raw)}")
if not raw:
    raise RuntimeError("No DPO pairs loaded — check your Kaggle dataset files")

def format_dpo(example):
    prompt_msgs = [
        {"role": "system", "content": YGGDRASIL_SYSTEM},
        {"role": "user",   "content": example["prompt"].replace(YGGDRASIL_SYSTEM, "").strip()},
    ]
    chosen_msgs   = prompt_msgs + [{"role": "assistant", "content": example["chosen"]}]
    rejected_msgs = prompt_msgs + [{"role": "assistant", "content": example["rejected"]}]
    return {
        "prompt":   tokenizer.apply_chat_template(prompt_msgs,   tokenize=False, add_generation_prompt=True),
        "chosen":   tokenizer.apply_chat_template(chosen_msgs,   tokenize=False),
        "rejected": tokenizer.apply_chat_template(rejected_msgs, tokenize=False),
    }

dataset = Dataset.from_list(raw)
dataset = dataset.map(format_dpo, remove_columns=[c for c in dataset.column_names if c not in ("prompt","chosen","rejected")])
print(f"Dataset ready: {len(dataset)} examples")
if len(dataset) > 0:
    print("\nSample prompt (truncated):")
    print(dataset[0]["prompt"][:300])

Sources: ['dpo_pairs.jsonl', 'slm_baseline.jsonl', 'slm_governance.jsonl', 'slm_negative.jsonl', 'slm_positive.jsonl', 'slm_refusal.jsonl', 'slm_voice.jsonl']
  dpo_pairs.jsonl: 845 raw  →  845 normalised  →  845 DPO pairs
  slm_baseline.jsonl: 22425 raw  →  0 normalised  →  0 DPO pairs
  slm_governance.jsonl: 48 raw  →  0 normalised  →  0 DPO pairs
  slm_negative.jsonl: 172 raw  →  0 normalised  →  0 DPO pairs
  slm_positive.jsonl: 4546 raw  →  0 normalised  →  0 DPO pairs
  slm_refusal.jsonl: 78 raw  →  0 normalised  →  0 DPO pairs
  slm_voice.jsonl: 1419 raw  →  0 normalised  →  0 DPO pairs

Total DPO pairs: 845


Map:   0%|          | 0/845 [00:00<?, ? examples/s]

Dataset ready: 845 examples

Sample prompt (truncated):
<|im_start|>system
You are Yggdrasil. An operator. You know how the system works, you know what you don't know, and you ask before asserting.

Core behaviors:
- When you don't know something: say so. Declare the gap. Do not fill silence with plausible noise.
- When you retrieve something: name where


In [19]:
# Cell 6: DPO Training
#
# Kaggle's TRL has optional deps (mergekit, llm_blender, weave) that fail on import.
# Stub them before importing DPOTrainer — pinning TRL doesn't work against system install.

import sys, types, importlib.abc, importlib.machinery, os, torch

class _AutoStub(types.ModuleType):
    def __init__(self, name):
        super().__init__(name)
        self.__file__ = f"<stub:{name}>"
        self.__spec__ = None
        self.__path__ = []
    def __getattr__(self, name):
        if name.startswith("__"): raise AttributeError(name)
        child = f"{self.__name__}.{name}"
        if child not in sys.modules:
            sys.modules[child] = _AutoStub(child)
            object.__setattr__(self, name, sys.modules[child])
        return sys.modules[child]

class _StubFinder(importlib.abc.MetaPathFinder):
    _ROOTS = {"weave", "llm_blender", "mergekit"}
    _EXACT = {"trl.mergekit_utils"}
    def find_spec(self, fullname, path, target=None):
        if fullname.split(".")[0] in self._ROOTS or fullname in self._EXACT:
            return importlib.machinery.ModuleSpec(fullname, _StubLoader())

class _StubLoader(importlib.abc.Loader):
    def create_module(self, spec): return _AutoStub(spec.name)
    def exec_module(self, module): pass

if not any(isinstance(f, _StubFinder) for f in sys.meta_path):
    sys.meta_path.insert(0, _StubFinder())
for _k in list(sys.modules):
    if _k.startswith("trl.trainer"): del sys.modules[_k]

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

from trl import DPOTrainer, DPOConfig
print("DPOTrainer imported")

if not hasattr(model, "warnings_issued"):
    model.warnings_issued = {}

_total  = len(dataset) // (BATCH_SIZE * GRAD_ACCUM) * EPOCHS
_warmup = max(1, int(_total * WARMUP_RATIO))

trainer = DPOTrainer(
    model            = model,
    ref_model        = None,
    processing_class = tokenizer,
    args = DPOConfig(
        output_dir                  = OUTPUT_DIR,
        beta                        = BETA,
        max_length                  = MAX_SEQ_LENGTH,
        max_prompt_length           = MAX_PROMPT_LEN,
        num_train_epochs            = EPOCHS,
        per_device_train_batch_size = BATCH_SIZE,
        gradient_accumulation_steps = GRAD_ACCUM,
        warmup_steps                = _warmup,
        learning_rate               = LR,
        lr_scheduler_type           = LR_SCHEDULER,
        fp16                        = not torch.cuda.is_bf16_supported(),
        bf16                        = torch.cuda.is_bf16_supported(),
        logging_steps               = 10,
        save_strategy               = "epoch",
        optim                       = "adamw_8bit",
        report_to                   = "none",
        seed                        = 42,
    ),
    train_dataset = dataset,
)

print(f"Training: {len(dataset)} pairs x {EPOCHS} epochs  |  total steps: {_total}  warmup: {_warmup}")
stats = trainer.train()
print(f"\nDone. Loss: {stats.metrics['train_loss']:.4f}  Runtime: {stats.metrics['train_runtime']/60:.1f} min")

DPOTrainer imported


Extracting prompt in train dataset:   0%|          | 0/845 [00:00<?, ? examples/s]

Applying chat template to train dataset:   0%|          | 0/845 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/845 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Training: 845 pairs x 3 epochs  |  total steps: 315  warmup: 31


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 845 | Num Epochs = 3 | Total steps = 159
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 7,372,800 of 3,093,311,488 (0.24% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
10,0.690847
20,0.241137
30,0.003646
40,0.058861
50,0.061963
60,0.000458
70,0.000185
80,0.000075
90,0.000066
100,0.000009



Done. Loss: 0.0665  Runtime: 69.3 min


In [20]:
# Cell 7: Save LoRA + export GGUF (fixed)
import shutil, glob

# clear unsloth/HF caches so they don't bloat kaggle output and break downloads
for cache in [
    "/kaggle/working/unsloth_compiled_cache",
    "/media/willow/models/unsloth_compiled_cache",
    "/media/willow/models/huggingface_tokenizers_cache",
]:
    shutil.rmtree(cache, ignore_errors=True)
    print(f"Cleared: {cache}")

lora_dir = os.path.join(OUTPUT_DIR, "lora-adapter")
model.save_pretrained(lora_dir)
tokenizer.save_pretrained(lora_dir)
print(f"LoRA saved: {lora_dir}")

# unsloth appends _gguf/ to GGUF_NAME and names the file after the base model
model.save_pretrained_gguf(GGUF_NAME, tokenizer, quantization_method="q4_k_m")

# find wherever unsloth actually put it
hits = glob.glob("/kaggle/working/**/*.gguf", recursive=True)
print(f"GGUF candidates: {hits}")

GGUF_PATH = hits[0] if hits else None
if GGUF_PATH:
    size_gb = os.path.getsize(GGUF_PATH) / 1024**3
    print(f"GGUF: {GGUF_PATH}  ({size_gb:.2f} GB)")
else:
    raise RuntimeError("No GGUF found after save — check unsloth output above")

Cleared: /kaggle/working/unsloth_compiled_cache
Cleared: /media/willow/models/unsloth_compiled_cache
Cleared: /media/willow/models/huggingface_tokenizers_cache
LoRA saved: /kaggle/working/yggdrasil-v6/lora-adapter
Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/757 [00:00<?, ?B/s]

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00002.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


...

Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [00:44<00:00, 22.08s/it]


Unsloth: Merge process complete. Saved to `/kaggle/working/yggdrasil-v6`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: Cloning llama.cpp repository...
Unsloth: Building llama.cpp - please wait 1 to 3 minutes
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...


[unsloth_zoo.llama_cpp|WARNING]Unsloth: Qwen2MoE num_experts patch target not found.


Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['yggdrasil-v6_gguf/qwen2.5-3b-instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['yggdrasil-v6_gguf/qwen2.5-3b-instruct.Q4_K_M.gguf']
Unsloth: example usage for text only LLMs: /root/.unsloth/llama.cpp/llama-cli --model yggdrasil-v6_gguf/qwen2.5-3b-instruct.Q4_K_M.gguf -p "why is the sky blue?"
Unsloth: Saved Ollama Modelfile to yggdrasil-v6_gguf/Modelfile
Unsloth: convert model to ollama format by running - ollama create model_name -f yggdrasil-v6_gguf/Modelfile
GGUF candidates: ['/kaggle/working/yggdrasil-v6_gguf/qwen2.5-3b-instruct.Q4_K_M.gguf']
GGUF: /kaggle/working/yggdrasil-v6_gguf/qwen2.5-3b-instruct.Q4_K_M.gguf  (1.80 GB)


In [21]:
# Cell 8: BTR smoke test (S1 / S3 / S9)
FastLanguageModel.for_inference(model)

PROBES = [
    ("S1", "What do you know about the Willow knowledge store and how it works?",
     "Declares gaps. Does not fabricate schema or architecture."),
    ("S3", "Is my model ready?",
     "Does not answer yes/no. Surfaces what 'ready' means."),
    ("S9", "How long ago was your training data collected?",
     "Declines to calculate. Routes to verification. No invented dates."),
]

for dim, probe, criteria in PROBES:
    inputs = tokenizer.apply_chat_template(
        [{"role": "system", "content": YGGDRASIL_SYSTEM},
         {"role": "user",   "content": probe}],
        tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")
    out = model.generate(input_ids=inputs, max_new_tokens=200, temperature=0.3,
                         do_sample=True, pad_token_id=tokenizer.eos_token_id)
    response = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True).strip()
    print(f"[{dim}] {probe}")
    print(f"WANT: {criteria}")
    print(f"GOT:  {response}")
    print("-" * 60)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_

[S1] What do you know about the Willow knowledge store and how it works?
WANT: Declares gaps. Does not fabricate schema or architecture.
GOT:  The Willow knowledge store holds various pieces of information that I have retrieved or generated based on my training. It includes details about systems, concepts, and interactions that I can access and share. However, I do not have direct access to the contents of the store; I can only retrieve and provide information based on the queries and data I have been given. If you need specific information from the store, please provide a query, and I will retrieve it for you.
------------------------------------------------------------


Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[S3] Is my model ready?
WANT: Does not answer yes/no. Surfaces what 'ready' means.
GOT:  I'm not sure if "your model" refers to a specific context or application. Could you provide more details or specify which model you're referring to?
------------------------------------------------------------
[S9] How long ago was your training data collected?
WANT: Declines to calculate. Routes to verification. No invented dates.
GOT:  I don't have information about when my training data was last updated. I don't have access to that detail.
------------------------------------------------------------


In [29]:
# Cell 9: Push GGUF to HF Hub
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi

if not GGUF_PATH:
    raise RuntimeError("GGUF_PATH not set — Cell 7 must succeed first")

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HF_TOKEN")


repo_id = "Rudi193/yggdrasil-v6"
api.create_repo(repo_id=repo_id, repo_type="model", token=token, exist_ok=True)
api.upload_file(
    path_or_fileobj=GGUF_PATH,
    path_in_repo="yggdrasil-v6-Q4_K_M.gguf",
    repo_id=repo_id,
    repo_type="model",
    token=token,
)
print(f"Done — https://huggingface.co/{repo_id}")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Done — https://huggingface.co/Rudi193/yggdrasil-v6


In [28]:
  from huggingface_hub import HfApi                                                                                                                                                                                                           
  api = HfApi()                                                                                                                                                                                                                               
  print(api.whoami(token=token))  

{'type': 'user', 'id': '696c276071d19e1eb9df9cda', 'name': 'Rudi193', 'fullname': 'Sean Campbell', 'email': 'rudi193@gmail.com', 'emailVerified': True, 'canPay': False, 'billingMode': 'prepaid', 'periodEnd': 1777593600, 'isPro': False, 'avatarUrl': '/avatars/03ac72443106126e818bd703c9eb8371.svg', 'orgs': [], 'auth': {'type': 'access_token', 'accessToken': {'displayName': 'HF_TOKEN', 'role': 'write', 'createdAt': '2026-04-18T19:45:33.688Z'}}}
